In [ ]:
# 1) GPU check + package installs
import sys
print(sys.version)

!nvidia-smi
!pip -q install pretty_midi mido tqdm

In [ ]:
# pyright: reportMissingImports=false, reportMissingModuleSource=false
# 2) All imports + config values (standard Task 3 settings)
import random
import shutil
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

SEED = 42

# Use standard src/config-style Task 3 values for consistency across tasks.
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
NUM_EPOCHS = 30
VALIDATION_SPLIT = 0.2
NUM_WORKERS = 0

TOKEN_PAD_ID = 0
TOKEN_BOS_ID = 1
TOKEN_EOS_ID = 2
TOKEN_NOTE_OFFSET = 3
TOKEN_SEQUENCE_LENGTH = 512
TOKEN_VOCAB_SIZE = 91

TR_MODEL_DIM = 256
TR_NUM_HEADS = 8
TR_NUM_LAYERS = 6
TR_FF_DIM = 1024
TR_DROPOUT = 0.1
TR_LABEL_SMOOTHING = 0.05
TR_NUM_SAMPLES = 10
TR_GENERATION_MAX_TOKENS = 512

LAKH_DATASET_CANDIDATES = [
    Path('/kaggle/input/lakh-midi-clean'),
    Path('/kaggle/input/datasets/imsparsh/lakh-midi-clean'),
    Path('/kaggle/input/lakh-midi-dataset'),
    Path('/kaggle/input/lakh-midi'),
    Path('/kaggle/input/lmd-full'),
    Path('/kaggle/input/lmd-matched'),
]

DATASET_PATH = None
for candidate in LAKH_DATASET_CANDIDATES:
    if candidate.exists():
        DATASET_PATH = candidate
        break

if DATASET_PATH is None:
    raise FileNotFoundError(
        'LAKH dataset path not found. Attach LAKH in Kaggle and update LAKH_DATASET_CANDIDATES if needed.'
    )

OUTPUT_ROOT = Path('/kaggle/working/outputs')
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'
PLOTS_DIR = OUTPUT_ROOT / 'plots'
GENERATED_MIDI_DIR = OUTPUT_ROOT / 'generated_midis'

for directory in [OUTPUT_ROOT, CHECKPOINTS_DIR, PLOTS_DIR, GENERATED_MIDI_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Task 3 dataset path:', DATASET_PATH)

CODE_INPUT_CANDIDATES = [
    Path('/kaggle/input/music-generation-unsupervised-task3-src'),
    Path('/kaggle/input/music-generation-unsupervised-task2-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task3-src'),
    Path('/kaggle/input/datasets/farhantaukir/music-generation-unsupervised-task2-src'),
]

code_input_root = None
for candidate in CODE_INPUT_CANDIDATES:
    if candidate.exists():
        code_input_root = candidate
        break

if code_input_root is None:
    raise FileNotFoundError(
        'Code dataset path not found. Run "!ls /kaggle/input" and update CODE_INPUT_CANDIDATES.'
    )

if (code_input_root / 'music-generation-unsupervised').exists():
    code_input_root = code_input_root / 'music-generation-unsupervised'

PROJECT_ROOT = Path('/kaggle/working/music-generation-unsupervised')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(code_input_root, PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print('Code input used:', code_input_root)
print('Project root exists:', PROJECT_ROOT.exists())

In [ ]:
# pyright: reportMissingImports=false
# 3) Data loading and preprocessing
from src.training.train_transformer import build_dataloaders

train_loader, validation_loader = build_dataloaders(
    data_root=DATASET_PATH,
    chunk_length=TOKEN_SEQUENCE_LENGTH,
    split_name='task3_lakh',
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

print('Data root used:', DATASET_PATH)
print('Using all discovered files (no limit).')
print(f'Train sequences: {len(train_loader.dataset)}')
print(f'Validation sequences: {len(validation_loader.dataset)}')

In [ ]:
# 4) Model definition (from src/models/transformer.py)
from src.models.transformer import MusicTransformer

model = MusicTransformer(
    vocab_size=TOKEN_VOCAB_SIZE,
    d_model=TR_MODEL_DIM,
    nhead=TR_NUM_HEADS,
    num_layers=TR_NUM_LAYERS,
    dim_feedforward=TR_FF_DIM,
    dropout=TR_DROPOUT,
    pad_id=TOKEN_PAD_ID,
).to(DEVICE)

model

In [ ]:
# 5) Training loop (from src/training/train_transformer.py)
from src.training.train_transformer import train_transformer

results = train_transformer(
    train_loader=train_loader,
    validation_loader=validation_loader,
    model=model,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    label_smoothing=TR_LABEL_SMOOTHING,
)

print('Best checkpoint saved at:', results['checkpoint_path'])
print('Last validation loss:', results['validation_losses'][-1])
print('Last validation perplexity:', results['validation_perplexities'][-1])

In [ ]:
# 6) Perplexity plot save -> /kaggle/working/outputs/plots/task3_perplexity_curve.png
perplexity_plot_path = PLOTS_DIR / 'task3_perplexity_curve.png'

plt.figure(figsize=(10, 5))
plt.plot(results['validation_perplexities'], label='Validation Perplexity')
plt.xlabel('Epoch')
plt.ylabel('Perplexity')
plt.title('Task 3 - Transformer Validation Perplexity')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(perplexity_plot_path, dpi=200)
plt.show()

print('Saved:', perplexity_plot_path)

In [ ]:
# 7) Music generation -> save 10 MIDIs
from src.generation.generate_music import generate_task3_samples

generated_paths = generate_task3_samples(
    checkpoint_path=results['checkpoint_path'],
    num_samples=TR_NUM_SAMPLES,
    max_new_tokens=TR_GENERATION_MAX_TOKENS,
    output_dir=GENERATED_MIDI_DIR,
)

for path in generated_paths:
    print(path)

In [ ]:
# 8) Summary cell printing all deliverables
expected_files = [
    PLOTS_DIR / 'task3_perplexity_curve.png',
] + [GENERATED_MIDI_DIR / f'task3_sample_{index}.mid' for index in range(1, 11)]

print('Task 3 Deliverables Summary')
for file_path in expected_files:
    status = 'OK' if file_path.exists() else 'MISSING'
    print(f'[{status}] {file_path}')